# CLIP model implementation in PyTorch

## import dependencies

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import math

## Supper Parameters for the model (can be adjusted as needed)

In [ ]:
class Config:
    image_resolution = 256
    hidden_dim = 512
    output_dim = 256
    num_heads = 8
    dropout = 0.1
    text_encoder_layers = 2  # Number of transformer layers for the text encoder
    image_encoder_layers = 4  # Number of transformer layers for the image encoder
    vocab_size = 100  # Example vocabulary size, adjust as needed

## Image PatchEmbedding (VIT)

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, image_size, patch_size, in_channels, embed_dim):
        super(PatchEmbedding, self).__init__()
        self.image_size = image_size
        self.patch_size = patch_size
        self.num_patches = (image_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)  # (B, embed_dim, H/patch_size, W/patch_size)
        x = x.flatten(2)  # (B, embed_dim, num_patches)
        x = x.transpose(1, 2)  # (B, num_patches, embed_dim)
        return x

## Transformer block (for potential use in both image and text encoders)

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_hidden_dim, dropout=0.1):
        super(TransformerBlock, self).__init__()
        self.attention = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_hidden_dim),
            nn.ReLU(),
            nn.Linear(ff_hidden_dim, embed_dim)
        )
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        attn_output, _ = self.attention(x, x, x)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.ff(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

## Image encoder by Vision Transformer (ViT) style
## Image encoder: ViT patch embeddings → Transformer → CLS token.

In [ ]:
class ImageEncoder(nn.Module):
    def __init__(self, image_resolution, hidden_dim, output_dim):
        super(ImageEncoder, self).__init__()
        # cls_token for Vision Transformer
        self.cls_token = nn.Parameter(torch.zeros(1, 1, hidden_dim))
        self.patch_embedding = PatchEmbedding(image_size=image_resolution, patch_size=16, in_channels=3, embed_dim=hidden_dim)
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim=hidden_dim, num_heads=Config.num_heads, ff_hidden_dim=hidden_dim*4, dropout=Config.dropout)
            for _ in range(Config.image_encoder_layers)  # Assuming 6 layers for the image encoder
        ])
        # Projection heads: map both(CLS token for image and EOT token for text) representations into the same embedding space.
        self.projection = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        """
        Forward pass for the image encoder.

        Args:
            x (torch.Tensor): Input image tensor of shape (B, 3, H, W)

        Returns:
            torch.Tensor: Encoded image features of shape (B, output_dim)
        """
        x = self.patch_embedding(x)
        B, N, C = x.shape
        cls_tokens = self.cls_token.expand(B, -1, -1)  # (B, 1, hidden_dim)
        x = torch.cat((cls_tokens, x), dim=1)  # (B, 1 + num_patches, hidden_dim)
        x = x.transpose(0, 1)  # (1 + num_patches, B, embed_dim) for transformer
        for block in self.blocks:
            x = block(x)
        x = x[0]  # Take the cls_token output
        x = self.projection(x)
        return x

## Text encoder by transformer (similar to image encoder)
## Text encoder: token embeddings → causal Transformer → EOT token.

In [ ]:
class TextEncoder(nn.Module):
    def __init__(self, vocab_size, hidden_dim, output_dim):
        super(TextEncoder, self).__init__()
        # eot_token for text transformer (similar to cls_token in image transformer)
        self.eot_token = nn.Parameter(torch.zeros(1, 1, hidden_dim))
        # vocabulary embedding for text input
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim=hidden_dim, num_heads=Config.num_heads, ff_hidden_dim=hidden_dim*4, dropout=Config.dropout)
            for _ in range(Config.text_encoder_layers)  # Assuming 6 layers for the text encoder
        ])
        # Projection heads: map both(CLS token for image, EOT token for text) representations into the same embedding space.
        self.projection = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        """
        Forward pass for the text encoder.

        Args:
            x (torch.Tensor): Input text tensor of shape (B, seq_len)

        Returns:
            torch.Tensor: Encoded text features of shape (B, output_dim)
        """
        x = self.embedding(x)  # (B, seq_len, hidden_dim)
        B, N, C = x.shape
        eot_tokens = self.eot_token.expand(B, -1, -1)  # (B, 1, hidden_dim)
        x = torch.cat((x, eot_tokens), dim=1)  # (B, seq_len + 1, hidden_dim)
        x = x.transpose(0, 1)  # (seq_len + 1, B, embed_dim) for transformer
        for block in self.blocks:
            x = block(x)
        x = x[-1]  # Take the eot_token output
        x = self.projection(x)
        return x

## CLIP model combining image and text encoders

In [ ]:
class CLIP(nn.Module):
    def __init__(self, image_input_dim, vocab_size, hidden_dim, output_dim):
        super(CLIP, self).__init__()
        self.image_encoder = ImageEncoder(image_resolution=Config.image_resolution, hidden_dim=hidden_dim, output_dim=output_dim)
        self.text_encoder = TextEncoder(vocab_size, hidden_dim, output_dim)
        # Learnable log inverse temperature.
        self.logit_scale = nn.Parameter(
            torch.tensor(math.log(1 / 0.07))
        )

    def forward(self, image, text):
        """
        Forward pass for the CLIP model.

        Args:
            image (torch.Tensor): Input image tensor of shape (B, 3, H, W)
            text (torch.Tensor): Input text tensor of shape (B, seq_len)

        Returns:
            tuple: A tuple containing:
                - image_features (torch.Tensor): Encoded image features of shape (B, output_dim)
                - text_features (torch.Tensor): Encoded text features of shape (B, output_dim)
                - similarity (torch.Tensor): Cosine similarity matrix of shape (B, B)
        """
        image_features = self.image_encoder(image)
        text_features = self.text_encoder(text)
        # cosine similarity between image and text features
        image_features = F.normalize(image_features, p=2, dim=1)
        text_features = F.normalize(text_features, p=2, dim=1)

        # cosine similarity as logits
        # [B, D] @ [D, B] -> [B, B]
        # The scale is used to amplify the similarity score between images and text, making it easier for the contrastive loss to distinguish positive pairs from negative pairs.
        # Since the features are normalized, dot-product similarity typically ranges from -1 to 1. Multiplying by an initial scale of approximately 14.29 widens the gap between scores; when softmax is subsequently applied to the logits, the probability distribution of positive pairs becomes more prominent. Without scaling, the scores may be too close, resulting in a flat softmax distribution, which makes it harder for the model to learn to match paired image-text samples and reject unpaired ones.
        # This scaling factor is learnable but capped by .clamp(max=100) to prevent logits from being scaled excessively. It is applied to the image-text similarity matrix.
        scale = self.logit_scale.exp().clamp(max=100)

        logits_per_image = (
            scale * image_features @ text_features.T
        )

        logits_per_text = logits_per_image.T
        return logits_per_image, logits_per_text

## Why using scale
```python
self.logit_scale = nn.Parameter(
        torch.tensor(math.log(1 / 0.07))
    )
scale = self.logit_scale.exp().clamp(max=100)
```

The `scale` parameter is used to amplify the similarity scores between images and text, making it easier for the contrastive loss to distinguish between correct and incorrect pairs.
Since the features are normalized, the dot-product similarity typically falls within the range of -1 to 1. Multiplying by an initial scale factor of approximately 14.29 widens the gap between scores; consequently, when the `softmax` function is applied to the logits, the probability distribution for correct pairs becomes more pronounced. Without this scaling, the scores might be too close to one another, <span style="color:red"><b>resulting in a "flat" softmax distribution—which makes it difficult for the model to learn to differentiate between matching and non-matching image-text pairs.</b></span>
This scaling factor is learnable, but it is capped at a maximum value of 100 (via `.clamp(max=100)`) to prevent the logits from becoming excessively large.

## Example usage of the CLIP model

In [ ]:
if __name__ == "__main__":
    model = CLIP(Config.image_resolution, Config.vocab_size, Config.hidden_dim, Config.output_dim)
    # batch size of 3 RGB images represented as (B, C, H, W)
    dummy_image = torch.randn(3, 3, Config.image_resolution, Config.image_resolution)
    # sequence length of 16 for text. The shape of dummy_text is (B, seq_len)
    dummy_text = torch.randint(0, Config.vocab_size, (3, 16))  # Assuming sequence length of 16
    logits_per_image, logits_per_text = model(dummy_image, dummy_text)
    print("Logits per image: \n", logits_per_image)
    print("Logits per text: \n", logits_per_text)
    print("Model output shapes:")
    print("Logits per image shape:", logits_per_image.shape)
    print("Logits per text shape:", logits_per_text.shape)
    print("Done.")